## Proyecto 2: agente de atención al cliente — aerolínea

Une todo lo de semana 2 en un solo flujo: **Gradio** como UI, **function calling** para consultar datos reales (nada inventado), y **Gemini** en vez de Ollama porque ya vimos que el modelo local chico ignora resultados de tools y alucina — acá necesitamos que el agente no invente precios.

Diferencia clave de arquitectura vs los ejemplos anteriores: usamos `client.chats.create()` en vez de `generate_content()` — mantiene el historial de la conversación adentro de la sesión, así no tenemos que reconstruir `messages` a mano en cada vuelta como hacíamos con Ollama.

In [8]:
from dotenv import load_dotenv
from google import genai
from google.genai import types
import os

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

def precio_vuelo(ciudad_destino: str) -> str:
    """Devuelve el precio de un vuelo a una ciudad destino."""
    precios = {"madrid": 850, "roma": 720, "tokio": 1400}
    precio = precios.get(ciudad_destino.lower(), "desconocido")
    return f"El vuelo a {ciudad_destino} cuesta {precio} USD"


`client.chats.create()` arma una sesión que recuerda toda la conversación sola — cada `send_message()` ya sabe lo que se dijo antes. El `system_instruction` fija el rol del agente una sola vez, no hace falta repetirlo en cada mensaje.

Le sumamos voz al agente: mismo patrón de síntesis que en `03-multimodal-tts` (`gemini-2.5-flash-preview-tts` + módulo `wave`), pero ahora la voz sale de la respuesta real del agente en cada turno, no de un texto fijo. Con `type="messages"` en `ChatInterface` podemos devolver texto y audio juntos en la misma burbuja.

In [10]:
import gradio as gr
import wave

chat_session = client.chats.create(
    model="gemini-flash-lite-latest",
    config=types.GenerateContentConfig(
        system_instruction="Sos un agente de atención al cliente de una aerolínea. Respondé corto y amable.",
        tools=[precio_vuelo],
    ),
)

def texto_a_audio(texto: str, path: str) -> str:
    """Sintetiza la respuesta del agente a voz (mismo patrón que 03-multimodal-tts)."""
    response = client.models.generate_content(
        model="gemini-2.5-flash-preview-tts",
        contents=texto,
        config=types.GenerateContentConfig(
            response_modalities=["AUDIO"],
            speech_config=types.SpeechConfig(
                voice_config=types.VoiceConfig(
                    prebuilt_voice_config=types.PrebuiltVoiceConfig(voice_name="Kore")
                )
            ),
        ),
    )
    audio = response.candidates[0].content.parts[0].inline_data
    with wave.open(path, "wb") as wav_file:
        wav_file.setnchannels(1)
        wav_file.setsampwidth(2)
        wav_file.setframerate(24000)
        wav_file.writeframes(audio.data)
    return path

def chat(message, history):
    response = chat_session.send_message(message)
    audio_path = texto_a_audio(response.text, f"respuesta_{len(history)}.wav")
    return [response.text, gr.Audio(audio_path)]

gr.ChatInterface(
    chat,
    title="Agente de aerolínea",
    description="Preguntame por precios de vuelos a Madrid, Roma o Tokio.",
).launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
